# 02 — Kafka/MSK streaming ingest (EMR)

Read clickstream events from a live Amazon MSK topic with Spark Structured Streaming and land them as an append-only bronze Delta table on S3.

**Requires:**
1. An MSK cluster reachable from the EMR cluster's VPC/subnets (see `infra/terraform/`).
2. A topic (default `retail-clickstream`) — created via `scripts/create_msk_topics.sh` or manually.
3. Producer traffic on that topic — run `scripts/seed_kafka_topic.py`, or point a real producer at it.
4. The `aws-msk-iam-auth` and `spark-sql-kafka-0-10` packages on the classpath -- the `%%configure` cell below adds both via `spark.jars.packages` (this is a genuine setup step neither EMR nor this notebook automates any other way).

If MSK isn't provisioned yet, this notebook raises a clear error rather than silently doing nothing — use `01_batch_lakehouse_bronze_silver_gold.ipynb` or `07_file_rate_streaming_fallback.ipynb` to keep working while infra is being set up.

Run the cell below first, before anything else -- it configures Delta Lake for this notebook's Spark session (EMR doesn't bundle Delta by default, unlike Databricks). `%%configure -f` must run before any other Spark code in this session.

In [ ]:
%%configure -f
{"conf": {"spark.jars.packages": "io.delta:delta-spark_2.12:3.1.0,org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0,software.amazon.msk:aws-msk-iam-auth:2.2.0", "spark.sql.extensions": "io.delta.sql.DeltaSparkSessionExtension", "spark.sql.catalog.spark_catalog": "org.apache.spark.sql.delta.catalog.DeltaCatalog"}}

In [ ]:
schema = "retail_lakehouse"
base_path = "s3://<your-lakehouse-bucket>/data"
kafka_bootstrap_servers = ""  # e.g. b-1.xxx.kafka.us-east-1.amazonaws.com:9098 -- from `terraform output -raw msk_bootstrap_brokers_command`
kafka_topic = "retail-clickstream"
starting_offsets = "earliest"

from retail_lakehouse.config import PipelineConfig
cfg = PipelineConfig(schema=schema, base_path=base_path,
                      kafka_bootstrap_servers=kafka_bootstrap_servers, kafka_topic=kafka_topic)
spark.sql(f"USE `{cfg.schema}`")

if not kafka_bootstrap_servers:
    raise ValueError("Set kafka_bootstrap_servers before running this notebook.")

## Configure the Kafka source

IAM auth (`AWS_MSK_IAM`) uses the EMR instance profile's credentials ambiently -- no service-credential indirection needed, unlike the Databricks-serverless version of this notebook.

In [ ]:
from pyspark.sql import functions as F

reader = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", kafka_bootstrap_servers)
    .option("subscribe", kafka_topic)
    .option("startingOffsets", starting_offsets)
    .option("failOnDataLoss", "false")
    .option("maxOffsetsPerTrigger", 10000)
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "AWS_MSK_IAM")
    .option("kafka.sasl.jaas.config", "software.amazon.msk.auth.iam.IAMLoginModule required;")
    .option("kafka.sasl.client.callback.handler.class", "software.amazon.msk.auth.iam.IAMClientCallbackHandler")
)

raw_kafka = reader.load()

## Parse and write to bronze

`parse_kafka_value` (from `retail_lakehouse.transformations`) is unit-tested in `tests/test_kafka_parsing.py` against a synthetic Kafka-shaped DataFrame, so this logic is verified in CI without needing a live broker.

In [ ]:
from retail_lakehouse.transformations import parse_kafka_value, add_ingest_metadata

parsed = parse_kafka_value(raw_kafka)
bronze_stream = add_ingest_metadata(parsed, "kafka_msk")

query = (
    bronze_stream.writeStream
    .format("delta")
    .option("checkpointLocation", cfg.checkpoint("kafka_bronze_clickstream"))
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(cfg.table("bronze_clickstream_kafka"))
)
query.awaitTermination()

print("Micro-batch(es) complete. Query progress:")
for p in query.recentProgress[-5:]:
    print({"timestamp": p["timestamp"], "numInputRows": p["numInputRows"], "durationMs": p["durationMs"]})

spark.table(cfg.table("bronze_clickstream_kafka")).orderBy(F.desc("_ingest_ts")).limit(20).toPandas()

## Next

`03_streaming_silver_gold_delta.ipynb` reads `bronze_clickstream_kafka` as a stream and builds the silver/gold layers with watermarking, de-duplication, and an idempotent upsert into gold.